In [46]:
import numpy as np
import pandas as pd
from numpy import nan
from sklearn.preprocessing import  StandardScaler, OneHotEncoder
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA, TruncatedSVD
from nltk.stem import SnowballStemmer
from sklearn import linear_model, datasets
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


import nltk
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [47]:
df = pd.read_csv("winter_project_2026/development.csv")
eva= pd.read_csv("winter_project_2026/evaluation.csv")

In [48]:
rows = df.loc[df.duplicated(subset=['article', 'title'])]

#print(df.duplicated(subset=['article']).sum())
#print(df.duplicated(subset=['title']).sum())
#print(df.duplicated(subset=['title','article']).sum())

duplicated = df.loc[df.duplicated(subset=['title','article'])]

mask = df['article'].isin(duplicated['article'])
to_drop = df[mask]

df.drop(to_drop["Id"], inplace=True)
df.drop(["Id", 'timestamp'], inplace=True, axis=1)
df.dropna(inplace=True)

In [49]:
y = df['label']
df.drop('label', inplace=True, axis=1)

In [50]:
## unisco le colonne titolo e articolo in una colonna unica raddoppiando l'importanza del titolo
#text = df['title'] * 2 + df['article']
#df = pd.concat([df, text], axis=1)
#
##rinomino le colonne e droppo title e article
#df.rename(columns={0: 'text'}, inplace=True)
#df.drop(['title', 'article'], inplace=True, axis=1)

df.columns

Index(['source', 'title', 'article', 'page_rank'], dtype='object')

In [51]:
X_train, X_test, y_train, y_test = train_test_split(df, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y, shuffle=True)



In [52]:
stemmer = SnowballStemmer("english")


def stemmed_tokenizer(text):

    try:
        tokens = nltk.word_tokenize(text)
    except LookupError:
        tokens = text.split()
    stems = [stemmer.stem(t) for t in tokens if t.isalpha()]
    
    return stems

In [53]:
#ora faccio ohe sulla colonna source, scalo page_rank e vettorizzo la colonna text

encoder = OneHotEncoder(min_frequency=70, handle_unknown='infrequent_if_exist')
scaler = StandardScaler()
text_pipeline = Pipeline([
    ('vect', TfidfVectorizer(
        max_features=10000,
        stop_words='english',
        tokenizer=stemmed_tokenizer,
        ngram_range=(1, 2),
        max_df=0.6,
        min_df=5
    )),
    ('svd', TruncatedSVD(n_components=300, algorithm='arpack', random_state=RANDOM_SEED)) 
])

vectorizer1 = TfidfVectorizer(
        max_features=10000,
        stop_words='english',
        tokenizer=stemmed_tokenizer,
        ngram_range=(1, 2),
        max_df=0.7,
        min_df=5)
vectorizer2 = TfidfVectorizer(
        max_features=30000,
        stop_words='english',
        tokenizer=stemmed_tokenizer,
        ngram_range=(1, 2),
        max_df=0.9,
        min_df=5)

preprocessor = ColumnTransformer(transformers=[
    ('source', encoder, ['source']),
    ('page_rank', scaler, ['page_rank']),
    ('title',vectorizer1, 'title'),
    ('art',vectorizer2, 'article')
], remainder='drop', n_jobs=-1)


In [54]:
X_train = preprocessor.fit_transform(X_train)


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['abov', 'afterward', 'alon', 'alreadi', 'alway', 'ani', 'anoth', 'anyon', 'anyth', 'anywher', 'becam', 'becaus', 'becom', 'befor', 'besid', 'cri', 'describ', 'dure', 'els', 'elsewher', 'empti', 'everi', 'everyon', 'everyth', 'everywher', 'fifti', 'forti', 'henc', 'hereaft', 'herebi', 'howev', 'hundr', 'in

In [55]:
#C = [0.1, 1, 10]
#penalty = ['l1', 'l2']
#solver = ['liblinear', 'saga']
#
#
#hyperparameters = dict(C=C, penalty=penalty, solver=solver)
#logistic = linear_model.LogisticRegression()
#gridsearch = GridSearchCV(logistic, hyperparameters, n_jobs=-1)
#best_model = gridsearch.fit(X_train, y_train)
#
#print(best_model.best_estimator_)

In [56]:
clf = LogisticRegression(random_state=RANDOM_SEED,C=0.3, penalty='l2', solver='saga', class_weight='balanced', n_jobs=-1).fit(X_train, y_train)
X_test = preprocessor.transform(X_test)
y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred))

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['abov', 'afterward', 'alon', 'alreadi', 'alway', 'ani', 'anoth', 'anyon', 'anyth', 'anywher', 'becam', 'becaus', 'becom', 'befor', 'besid', 'cri', 'describ', 'dure', 'els', 'elsewher', 'empti', 'everi', 'everyon', 'everyth', 'everywher', 'fifti', 'forti', 'henc', 'hereaft', 'herebi', 'howev', 'hundr', 'inde', 'mani', 'meanwhil', 'moreov', 'nobodi', 'noon', 'noth', 'nowher', 'onc', 'onli', 'otherwis', 'ourselv', 'perhap', 'pleas', 'sever', 'sinc', 'sincer', 'sixti', 'someon', 'someth', 'sometim', 'somewher', 'themselv', 'thenc', 'thereaft', '

              precision    recall  f1-score   support

           0       0.82      0.65      0.73      4438
           1       0.71      0.80      0.75      1848
           2       0.85      0.78      0.81      2119
           3       0.57      0.54      0.56      1778
           4       0.80      0.92      0.86      1663
           5       0.47      0.56      0.51      2117
           6       0.55      0.84      0.67       554

    accuracy                           0.70     14517
   macro avg       0.68      0.73      0.70     14517
weighted avg       0.72      0.70      0.70     14517



In [57]:
#from xgboost import XGBClassifier
#
## Definisci il modello XGBoost
#clf = XGBClassifier(
#    n_estimators=1000,      # Numero di alberi
#    learning_rate=0.05,     # Impara piano per essere preciso
#    max_depth=6,            # Profondità degli alberi
#    subsample=0.8,          # Usa solo l'80% dei dati per ogni albero (evita overfitting)
#    colsample_bytree=0.8,   # Usa solo l'80% delle feature per albero
#    n_jobs=-1,              # Usa tutti i core
#    random_state=42,
#    tree_method="hist"      # Molto più veloce su dataset grandi
#    # Non serve class_weight='balanced', XGBoost gestisce meglio.
#    # Se vuoi forzarlo, usa scale_pos_weight ma è complesso per multiclasse.
#)
#
## Fit
#clf.fit(X_train, y_train)
#X_test = preprocessor.transform(X_test)
#y_pred = clf.predict(X_test)
#
#print(classification_report(y_test, y_pred))